# Graha semantic segmentation inference
This notebook runs Graha/Lunar-FM semantic segmentation on Pipeline-generated Lunar WAC datacubes. Input selection follows the same `DATA_DICT` format as `semantic_ibm_train.ipynb`.

The raw datacube helper canonicalizes WAC bands to VIS (5) followed by UV (2), so native Graha `vis-uv` normalization can be used even though the source GeoTIFF stores UV first.

# Setup

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
logging.getLogger('rasterio._env').setLevel(logging.ERROR)

import matplotlib.pyplot as plt
import torch

In [ ]:
# Run from lfm/notebooks, or adjust this for your HPC checkout.
repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from lfm.all_models.sem_seg import build_graha_notebook_configs
from lfm.all_models.all_tasks.experiments import resolve_inference_checkpoint
from lfm.all_models.all_tasks.data.normalization import load_terramind_pretraining_stats
from lfm.all_models.sem_seg.data_cube_inference import (
    load_and_configure_input_data,
    plot_inference_results,
    preprocess_datacubes,
    sliding_window_inference,
)
from lfm.all_models.all_tasks.graha_inference import GrahaLogitModel
from lfm.full_model.sem_seg import semantic_graha_components
print('Successfully imported LFM and Graha modules')

# User configuration
-  `INPUT_ROOT_DIR`: where to load datacubes from (changing this risks breaking the notebook!)
-  `GRAHA_PRETRAIN_DIR`: where to load graha modality info, configuration from (changing this risks breaking the notebook!)
-  `GRAHA_LIGHTNING_CHECKPOINT`: optional path to a Graha semantic segmentation `.ckpt` or `.pt` file. Leave blank to discover the newest checkpoint under `./outputs/semantic_seg_finetuning`.
-  `OUTPUT_DIR`: where to place output plots, etc from this notebook

- `DATA_DICT`: dataset-level dictionary that controls the data-specific parts of training. It defines the dataset directory, dataset modality, selected modalities, optional file matching, band selection, NoData policy values, and optional Graha input mode override.

- `DATA_DICT["dataset_name"]`: dataset name for readability. Has no effect on model/dataset functionality.

- `DATA_DICT["data_dir"]`: path to dataset root. This root should contain `train`, `val`, and `test` split folders.

- `DATA_DICT["dataset_modality"]`: stored chip layout. Supported values include `"wac"`, `"wac_static"`, `"nac"`, and `"nac_dtm"`; this controls default normalization behavior.

- `DATA_DICT["selected_modalities"]`: **DO NOT CHANGE FOR THIS NOTEBOOK**. Frontend modalities to load from the stored chips.

- `DATA_DICT["band_filters"]`: modality-local band selection. For WAC, `"vis": [0, 1, 2, 3, 4]` and `"uv": [0, 1]` selects all 7 stored WAC channels. For NAC PHO or DTM, use `[0]` because each modality is stored as a single band. Do not use empty lists to remove a modality; remove that modality from `selected_modalities` and omit its band filter.

- `DATA_DICT["excluded_nodata_values"]`: only used for datasets containing static data. Known NoData values to ignore in model ingestion/loss behavior so sentinel values do not become learnable image structure.

- `DATA_DICT`: same structure as the training notebook. 
    - `selected_modalities` controls which canonical WAC/static groups are passed to inference
    - `band_filters` uses indices local to each group.
    - For raw WAC datacubes, the canonical groups are `vis: [0..4]`, `uv: [0..1]`, and `static: [0..N-1]` after the helper's static-band filter. Native `vis-uv` normalization is used when static is not selected. A non-native static subset uses Graha's flexible fused `wac` input path.

In [ ]:
INPUT_ROOT_DIR = Path('/explore/nobackup/projects/lfm/model_inputs/inference/WAC_Processed_AOI')
GRAHA_PRETRAIN_DIR = Path('/explore/nobackup/projects/lfm/ibm_model_pretrain_dir_v2')
GRAHA_LIGHTNING_CHECKPOINT = ''  # Blank discovers the latest semantic experiment checkpoint.
OUTPUT_DIR = Path('./outputs/inference')

DATA_DICT = {
    'dataset_name': 'wac_static_inference',
    'data_dir': str(INPUT_ROOT_DIR),
    'dataset_modality': 'wac_static',
    'selected_modalities': ['vis', 'uv', 'static'],  # DO NOT CHANGE for inference!
    'band_filters': {
        'vis': [0, 1, 2, 3, 4],
        'uv': [0, 1],
        "static": [  # Keep all 63 static bands; remove individual indices here for ablation tests.
            0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
            10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
            20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
            30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
            40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
            50, 51, 52, 53, 54, 55, 56, 57, 58, 59,
            60, 61, 62,
        ],
    },
    'excluded_nodata_values': [
        -32768.0,
        -3.4028226550889045e38,
        -3.4028230607370965e38,
        -3.4028234663852886e38,
    ],
}

Non-configurable values, but values that need to be set early.

In [ ]:
GRAHA_LIGHTNING_CHECKPOINT = resolve_inference_checkpoint(
    GRAHA_LIGHTNING_CHECKPOINT,
    task_subdir='semantic_seg_finetuning',
    outputs_root=NOTEBOOK_DIR / 'outputs',
)
MODEL_NATIVE_SIZE = 256
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load and configure the input data

In [ ]:
input_data = load_and_configure_input_data(INPUT_ROOT_DIR, DATA_DICT, verbose=True)
images_raw = input_data['images_raw']
nodata_masks = input_data['nodata_masks']
file_pairs = input_data['file_pairs']
BAND_FILTER = input_data['band_filter']
n_channels = input_data['n_channels']
GRAHA_BACKEND_MODALITIES = input_data['backend_modalities']
GRAHA_INPUT_MODE = input_data['input_mode']
NORMALIZATION_MODALITY = input_data['normalization_modality']
print(f"Input modalities: {input_data['selected_modalities']}; channels: {n_channels}")
print(f'Graha backend: {GRAHA_BACKEND_MODALITIES}')

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
notebook_configs = build_graha_notebook_configs(
    output_dir=OUTPUT_DIR, data_root=INPUT_ROOT_DIR,
    graha_base_output_dir=OUTPUT_DIR, graha_pretrain_dir=GRAHA_PRETRAIN_DIR,
    graha_lightning_checkpoint=GRAHA_LIGHTNING_CHECKPOINT, data_dict=DATA_DICT,
    normalization_modality=NORMALIZATION_MODALITY,
    graha_input_modality_mode=GRAHA_INPUT_MODE,
    graha_backend_modalities=GRAHA_BACKEND_MODALITIES, validate_paths=False,
)
config = notebook_configs.experiment_config
graha_config = notebook_configs.graha_config
deps = notebook_configs.dependencies
if NORMALIZATION_MODALITY is not None:
    means, stds = load_terramind_pretraining_stats(
        graha_config.modality_info,
        normalization_modality=NORMALIZATION_MODALITY,
        band_filter=BAND_FILTER,
    )
else:
    means, stds = None, None

task_cls = semantic_graha_components.make_downstream_shape_segmentation_task_class(
    deps['LunarShapeSegmentationTask']
)
sample_batch = {'image': torch.zeros(1, n_channels, MODEL_NATIVE_SIZE, MODEL_NATIVE_SIZE)}
graha_task = semantic_graha_components.create_task(graha_config, task_cls, sample_batch).to(device)
semantic_graha_components.inspect_backbone(graha_task)
semantic_graha_components.load_lightning_checkpoint_state(graha_task, GRAHA_LIGHTNING_CHECKPOINT, 'Graha')
graha_task.eval()

model = GrahaLogitModel(graha_task).to(device).eval()
print('Successfully loaded Graha semantic checkpoint')

# Inference

This cell processes the data cubes and passes them through the model for inference. The output figure will be saved to `outputs/inference/graha_inference_viz.png`. 

In [ ]:
images_graha = preprocess_datacubes(images_raw, means=means, stds=stds, nodata_masks=nodata_masks)
preds, probabilities = sliding_window_inference(images_graha, model=model, device=device, target_size=MODEL_NATIVE_SIZE, n_channels=n_channels, nodata_masks=nodata_masks, debug=True)
fig = plot_inference_results(images_graha, preds, file_pairs, OUTPUT_DIR, n_channels, nodata_masks=nodata_masks)